# Wczesne wykrywanie ryzyka ponownego przyjęcia pacjenta do szpitala (hospital readmission) - case study

<div style="text-align: center;"><img src=".//Images//Hospital_scene.png" alt="zadanie" width="400" height="120" style="margin: 10px; "/></div>

**Cel projektu**

W systemach opieki zdrowotnej ponowne przyjęcia pacjentów do szpitala w krótkim czasie po wypisie (np. w ciągu 30 dni) są poważnym problemem. Powodują dodatkowe koszty, zajmują miejsca szpitalne i mogą świadczyć o niewłaściwej opiece lub nieprzestrzeganiu zaleceń przez pacjenta. Celem projektu jest stworzenie modelu klasyfikacyjnego, który przewidzi z dużym wyprzedzeniem, czy pacjent zostanie ponownie przyjęty do szpitala w ciągu 30 dni od wypisu.

---

**Dane źródłowe**

Zbiór danych (dostępny lokalnie) pochodzi z repozytorium UCI Machine Learning:
**"Diabetes 130-US hospitals for years 1999–2008"**  
[Link do zbioru](https://archive.ics.uci.edu/ml/datasets/diabetes)

- Liczba rekordów: ~100 000
- Źródło: dane z 130 szpitali w USA
- Opis: każdy wiersz to jeden pobyt pacjenta chorego na cukrzycę
- Atrybuty: dane demograficzne (wiek, płeć), długość pobytu, leki, zabiegi, diagnozy, liczba wizyt ambulatoryjnych itp.

---

**Zmienna docelowa**

Kolumna `readmitted` zawiera wartości:
- `'NO'` – brak ponownego przyjęcia
- `'>30'` – przyjęcie po ponad 30 dniach
- `'<30'` – przyjęcie w ciągu 30 dni

**Dla celów klasyfikacji binarnej przekształcamy:**
- `y = 1` jeśli `readmitted == '<30'`
- `y = 0` w przeciwnym razie

Tak zdefiniowana klasa pozytywna (`<30`) występuje **w ok. 10–11% przypadków**, co czyni problem **silnie niezbalansowanym**.

---

**Korzyści i zastosowania praktyczne**

Model taki może zostać wykorzystany np. przez szpitalne systemy informatyczne (HIS), by:
- objąć pacjentów z wysokim ryzykiem readmisji dodatkowymi programami edukacyjnymi,
- zapewnić im kontakt telefoniczny po wypisie,
- skierować ich na wizyty kontrolne,
- lepiej planować zasoby i łóżka.

W dłuższej perspektywie może to przynieść **niższe koszty**, **lepszą jakość opieki** i **mniejsze ryzyko powikłań** dla pacjentów.

---

**Zadanie**

Zbadaj dla zbioru:
   - wybrany algorytm bez balansowania
   - wybrany z `class_weight='balanced'`
     
---

# Rozwiązanie

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    make_scorer, f1_score, recall_score, precision_score,
    accuracy_score, balanced_accuracy_score, confusion_matrix, classification_report
)
from imblearn.metrics import geometric_mean_score
from sklearn.impute import KNNImputer

In [ ]:
pd.set_option('display.max_columns', None)        # pokazuj wszystkie kolumny
pd.set_option('display.expand_frame_repr', False) # nie łam wierszy na kilka linii

<span style="color: navy; font-weight: bold;">Wczytanie danych</span>

In [ ]:
df = pd.read_csv("./Data/diabetic_data.csv")

<span style="color: navy; font-weight: bold;">Wstępne czyszczenie</span>

In [ ]:
# Wstępne czyszczenie
df.drop(['encounter_id', 'patient_nbr'], axis=1, inplace=True)
df = df[df['readmitted'].isin(['NO', '<30'])].copy()
df['readmitted'] = (df['readmitted'] == '<30').astype(int)
df = df.loc[:, df.nunique() > 1]

# Zamiana '?' na NaN
df.replace('?', np.nan, inplace=True)

# Kodowanie kolumn kategorycznych numerycznie, z zachowaniem NaN
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = pd.factorize(df[col])[0]

# Kodowanie one-hot, 
# Wybór kodowania zależy też od modelu # np. RandomForest, XGBoost, KNN tolerują factorize; 
# LogisticRegression, SVM lepiej działają z one-hot, które jednak zwiększa liczbę kolumn - z 44 do 2296 kolumn w tym przypadku :D

# categorical_cols = df.select_dtypes(include='object').columns
# df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Imputacja braków metodą KNN
imputer = KNNImputer(n_neighbors=5)
df_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

# Zaokrąglenie (opcjonalne)
df = df_imputed.round(0).astype(int)

<span style="color: navy; font-weight: bold;">Przygotowanie danych dla modelu wraz ze skalowaniem</span>

In [ ]:
# Podział danych
X = df.drop(columns='readmitted')
y = df['readmitted']
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# Skalowanie
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

<span style="color: red; font-weight: bold;">Zaproponuj modele</span>

<span style="color: red; font-weight: bold;">Zaproponuj metryki</span>

<span style="color: red; font-weight: bold;">Zaproponuj kroswalidację</span>

<span style="color: red; font-weight: bold;">Wykonaj obliczenia</span>

<span style="color: red; font-weight: bold;">Podsumuj wyniki</span>

<span style="color: red; font-weight: bold;">Zinterpretuj wyniki i napisz wnioski</span>